# 2. EchoNext: download, cohort, labels and five-fold design

**Primary cohort.** EchoNext (version 1.1.1) contains 100,000 12-lead ECGs from 36,286 adults at NewYork-Presbyterian/Columbia University Irving Medical Center, each recorded within one year before a transthoracic echocardiogram, with a composite structural heart disease (SHD) label and its 11 echocardiographic components. The waveforms are released at 250 Hz for 10 s after median filtering, clipping and standardisation of each lead.

This notebook builds the table of all ECGs, the patient-level five-fold design used by every later analysis, and the aggregate characteristics of Table 1.

## 2.1 Download the data (once, by hand)

EchoNext is a **restricted-access** PhysioNet project: you need a PhysioNet account and must sign its data use agreement on the project page.

* Project page: https://physionet.org/content/echonext/1.1.1/ (DOI [10.13026/7cfw-d091](https://doi.org/10.13026/7cfw-d091))
* Sign the data use agreement on that page (section "Files"), then download in a terminal, **inside `DATA_DIR`** (see notebook 00):

```bash
wget -r -N -c -np --user <your PhysioNet username> --ask-password https://physionet.org/files/echonext/1.1.1/
```

This creates `DATA_DIR/physionet.org/files/echonext/1.1.1/`, the default `ECHONEXT_DIR`. Files used (about 24 GB):

| File | Content |
|---|---|
| `EchoNext_metadata_100k.csv` | one row per ECG: patient key, split, race and ethnicity, sex, age, ECG measurements, echocardiographic values and labels |
| `EchoNext_train_waveforms.npy`, `..._val_...`, `..._test_...`, `..._no_split_...` | waveforms, N × 1 × 2500 × 12 (rows in the order of the metadata rows of that split) |

The `..._tabular_features.npy` files are not needed. If you downloaded the ZIP file instead, set `ECHONEXT_DIR` in notebook 00 to the unzipped folder.

In [ ]:
%run ./00_settings.ipynb

meta = echonext_metadata()
print(f"metadata: {len(meta):,} ECGs, {meta.patient_key.nunique():,} patients")
for s in SPLITS:
    w = echonext_waveforms(s)
    assert w.shape[0] == (meta.split == s).sum(), s
    print(f"  {s:<9} {w.shape[0]:>6,} ECGs, waveform array {w.shape} {w.dtype}")

## 2.2 All ECGs in one table

The four split files are concatenated in the order train, val, test, no_split; this **global row order** is also the row order of the embeddings (notebook 04). The released patient-level split is **not** used for the analysis: all patients enter the five-fold cross-fitting below, so that every patient is evaluated and the subgroups are as large as possible.

Each patient's most recent ECG (`most_recent_ecg` = 1) is the ECG on which the patient is evaluated; all ECGs of a patient are used for fitting when the patient is in a fitting set.

In [ ]:
A = echonext_rows(meta)
A["mr"] = A["most_recent_ecg"].astype(int) == 1
lat = A[A.mr]
assert lat.patient_key.is_unique and len(lat) == A.patient_key.nunique()
print(f"{len(A):,} ECGs, {len(lat):,} patients")
print(pd.DataFrame({"patients": lat.race_ethnicity.value_counts(), "ECGs": A.race_ethnicity.value_counts()}).loc[GROUPS6])

## 2.3 Patient-level five-fold design (Supplementary Fig. 2)

* Patients are divided into five folds, stratified by race and ethnicity (six categories) × composite SHD on the most recent ECG.
* In round *k*, fold *k* (20% of patients) is evaluated; of the other patients, 15% (12% overall) form the validation set and the rest (68%) the fitting set, with the same stratification.
* Fitting uses every ECG of the fitting patients; validation and evaluation use the most recent ECG.

The same design (same seeds) is used for every analysis of this cohort. Saved to `WORK_DIR/echonext/design.npz` (row level).

In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

pats = lat.patient_key.to_numpy()
strata = (lat.race_ethnicity.astype(str) + "_" + lat[E_COMP].astype(int).astype(str)).to_numpy()
fold_of = np.empty(len(pats), int)
for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED_FOLDS).split(pats, strata)):
    fold_of[te] = k
role = np.zeros((N_FOLDS, len(pats)), np.int8)                     # 0 fitting, 1 validation, 2 evaluation
for k in range(N_FOLDS):
    rest = np.flatnonzero(fold_of != k)
    _, va = next(StratifiedShuffleSplit(1, test_size=VAL_FRAC, random_state=SEED_FOLDS + 1 + k).split(rest, strata[rest]))
    role[k, rest[va]] = 1
    role[k, fold_of == k] = 2
D = {"pi_of_row": A.patient_key.map(pd.Series(np.arange(len(pats)), index=pats)).to_numpy(),   # patient index of every ECG
     "mr_row": A.mr.to_numpy(), "Y_row": A[E_LABELS].to_numpy(np.float64),                       # labels of every ECG
     "lat_row": lat.index.to_numpy(), "fold_of": fold_of, "role": role,
     "g": race_index(lat.race_ethnicity), "race6": lat.race_ethnicity.astype(str).to_numpy(),    # per patient (most recent ECG)
     "Y": lat[E_LABELS].to_numpy(np.float64), "age": lat.age_at_ecg.to_numpy(float),
     "male": (lat.sex.astype(str).str.lower() == "male").to_numpy(int), "loc": lat.location_setting.astype(str).to_numpy(),
     "year": lat.acquisition_year.to_numpy(int)}
np.savez_compressed(work("echonext") / "design.npz", **D)

design = {"n_patients": int(len(pats)), "n_ecgs": int(len(A)), "n_folds": N_FOLDS, "val_frac_of_rest": VAL_FRAC, "seed": SEED_FOLDS,
          "folds": [{"fold": k, "fit_patients": int((role[k] == 0).sum()), "val_patients": int((role[k] == 1).sum()),
                     "eval_patients": int((role[k] == 2).sum()), "fit_ecgs": int(len(fold_rows(D, k)[0])),
                     "eval_shd_pct": float(D["Y"][fold_of == k, 0].mean() * 100), "eval_black_pct": float((D["race6"][fold_of == k] == "black").mean() * 100)}
                    for k in range(N_FOLDS)]}
save_json(design, results("02_echonext") / "design.json")
pd.DataFrame(design["folds"]).round(1)

## 2.4 Characteristics (Table 1) and prevalence by group (Fig. 3c,d)

Age, sex and prevalence refer to the most recent ECG of each patient. The prevalence among **all** ECGs is also computed, because probes and erasers are fitted on all ECGs of the fitting patients; Fig. 3c,d uses it.

In [ ]:
rows = []
for name, d, dall in [("All", lat, A)] + [(gname, lat[lat.race_ethnicity == gname], A[A.race_ethnicity == gname]) for gname in GROUPS6]:
    r = {"cohort": "EchoNext", "group": name, "patients": len(d), "ecgs": len(dall), "age": median_iqr(d.age_at_ecg),
         "female_pct": 100 * (d.sex.astype(str).str.lower() != "male").mean()}
    for s in ("inpatient", "emergency", "outpatient", "procedural"):
        r[f"{s}_pct"] = 100 * (d.location_setting.astype(str) == s).mean()
    for lab in E_LABELS:
        r["prev_" + LABEL_NAME[lab]] = 100 * d[lab].mean()
        r["pos_" + LABEL_NAME[lab]] = int(d[lab].sum())
    rows.append(r)
characteristics = pd.DataFrame(rows)
characteristics.to_csv(results("02_echonext") / "characteristics.csv", index=False, encoding="utf-8-sig")

prev = pd.DataFrame(prevalence_rows("EchoNext", E_LABELS, D["Y"], D["g"], D["Y_row"], D["g"][D["pi_of_row"]]))
prev.to_csv(results("02_echonext") / "prevalence.csv", index=False, encoding="utf-8-sig")
characteristics[["group", "patients", "ecgs", "age", "female_pct", "prev_Composite SHD", "prev_LVEF ≤45%", "prev_Aortic stenosis", "pos_Aortic stenosis"]].round(1)